# 03c. Seq 모델 학습 (Walk-Forward)

## 📋 개요
GRU 등 시계열 전용 모델(SeqModelBase 계열)을 학습합니다.  
Tabular 트랙(03a)과 **완전히 독립**된 전용 트랙입니다.

## 🔄 Tabular 트랙(WFT)과의 차이

| 항목 | Tabular (03a) | Seq (03c) |
|------|--------------|----------|
| 입력 형태 | `(N, n_features)` 2D | `(N, seq_len, n_features)` 3D |
| 타겟 구성 | horizon별 shift | 슬라이딩 윈도우 자동 구성 |
| 피처 순서 정보 | 피처에 압축 | 시퀀스 구조로 직접 학습 |
| WalkForwardTrainer | 사용 | **사용 안 함** (SeqTrainer 전용) |

## ✅ 산출물
- `data/03_seq/{model_date}/gru/weights.pt`
- `data/03_seq/{model_date}/gru/config.json`
- `data/03_seq/{model_date}/gru/val_predictions.parquet` (n_folds=2인 경우만 생성)
- `data/03_seq/{model_date}/gru/test_predictions.parquet`

※ v4.0.0 메모리/IO 최적화: 파켓 저장 시 `fastparquet` 엔진을 사용하여 메타데이터 충돌을 방지합니다.

산출물 컬럼 규격이 Tabular 트랙과 동일하여 **05단계에서 구분 없이 평가 가능**합니다.

## 📌 v4.1.0 변경 사항
- **`integration_order` 지원**: `config.yaml`의 `integration_order` 값이
  `SeqTrainer`에 전달되어 log-close 역산 차수를 제어합니다.
  `order=2` 사용 시 02단계에서 생성된 `target_log_return_1d_lag1` 컬럼이 필요합니다.

## 🔧 Setup

In [ ]:
import warnings
import pandas as pd
import numpy as np

from src.utils.config import load_config, ProjectPaths, is_seq_model
from src.models.gru_model import GRUModel
from src.modeling.seq_trainer import SeqTrainer
from src.utils.export import save_ticker_csv, should_save_csv, load_ticker_name_map

warnings.filterwarnings('ignore')

In [ ]:
cfg      = load_config()
train_cfg = cfg['training']
seq_cfg   = cfg['sequence']

# 03c는 active_seq_model을 사용합니다.
# active_model(Tabular 트랙)과 독립적으로 운용됩니다.
active_seq = cfg.get('active_seq_model', 'gru')

if active_seq != 'gru':
    raise ValueError(
        f"active_seq_model='{active_seq}'은 현재 미지원입니다.\n"
        f"v4.0.0에서는 'gru'만 지원합니다."
    )

paths = ProjectPaths.from_config(cfg)
paths.ensure_dirs()

print(f"🚀 [Step 3c] Seq 모델 학습 시작")
print(f"   - 기준일:          {paths.reference_date}")
print(f"   - 모델:            {active_seq}")
print(f"   - seq_len:         {seq_cfg['seq_len']}")
print(f"   - forecast_horizon:{seq_cfg['forecast_horizon']}")
print(f"   - target_type:     {seq_cfg['target_type']}")
print(f"\n📂 경로:")
print(f"   - 입력:  {paths.get_dataset_parquet()}")
print(f"   - 모델:  {paths.get_seq_model_dir()}")

## 1️⃣ 데이터 로드

In [ ]:
print("📥 Loading dataset...")
df = pd.read_parquet(paths.get_dataset_parquet())
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

# 피처 컬럼 자동 추출
feature_cols = [c for c in df.columns if c.startswith('feature_')]

print(f"   - 행수:       {len(df):,}")
print(f"   - 종목 수:    {df['ticker'].nunique():,}")
print(f"   - 피처 수:    {len(feature_cols)}")
print(f"   - 날짜 범위:  {df['date'].min()} ~ {df['date'].max()}")

# target 컬럼 존재 확인
target_col = f"target_{seq_cfg['target_type'].replace('log_return_1d', 'log_return_1d')}"
# target_type이 log_return_1d이면 target_log_return_1d
target_col = 'target_log_return_1d' if seq_cfg['target_type'] == 'log_return_1d' \
             else 'target_log_close'

assert target_col in df.columns, (
    f"❌ '{target_col}' 컬럼이 없습니다. "
    f"02단계(02_build_dataset.ipynb)를 먼저 실행하세요."
)
print(f"   - 타겟 컬럼: {target_col} ✅")

## 2️⃣ 모델 및 Trainer 초기화

`config.yaml`의 `integration_order`가 `SeqTrainer`에 전달됩니다.
Tabular 트랙(03a)과 동일한 역산 차수를 사용하므로 두 트랙의 평가 결과가 동일 기준으로 비교됩니다.

In [ ]:
print("🧠 GRUModel 초기화...")
model = GRUModel(
    model_version    = f"v4.0.0_gru_{paths.reference_date}",
    params           = cfg['gru_params'],
    seq_len          = seq_cfg['seq_len'],
    forecast_horizon = seq_cfg['forecast_horizon'],
    feature_list     = feature_cols,
    target_type      = seq_cfg['target_type'],
    checkpoint_dir   = paths.get_seq_model_dir(),  # 개선 시 checkpoint 저장
)

print("🔧 SeqTrainer 초기화...")
trainer = SeqTrainer(
    model            = model,
    feature_cols     = feature_cols,
    target_col       = target_col,
    seq_len          = seq_cfg['seq_len'],
    forecast_horizon = seq_cfg['forecast_horizon'],
    stride           = seq_cfg.get('stride', 1),
    date_col         = 'date',
    integration_order=cfg.get('integration_order', 1),   # ← 추가
)

print(f"   target_columns[:3]: {model.target_columns[:3]}")

## 3️⃣ Walk-Forward 학습 실행

Tabular 트랙(03a)과 동일한 `train_end`, `valid_window_days`, `test_window_days`를 사용합니다.  
두 트랙의 평가 기간이 일치하여 성능 비교가 공정하게 이루어집니다.

In [ ]:
print("🏃 Seq Walk-Forward Training 실행 중...")

results = trainer.run(
    df                = df,
    train_end         = train_cfg['train_end'],
    valid_window_days = train_cfg['valid_window_days'],
    test_window_days  = train_cfg['test_window_days'],
    n_folds           = 1,   # 1-Fold 기본. 앙상블 필요 시 2로 변경(v4.1.0)
    resume            = False,  # True: checkpoint.pt에서 이어서 학습
    fit_kwargs        = {
        'epochs'  : cfg['gru_params'].get('epochs',   100),
        'patience': cfg['gru_params'].get('patience',  10),
    },
)

## 4️⃣ 결과 저장

In [ ]:
print("\n💾 저장 중...")

seq_model_dir = paths.get_seq_model_dir()

# 1. 모델 (weights.pt + config.json)
results['final_model'].save(str(seq_model_dir))
print(f"   ✅ 모델 저장: {seq_model_dir}")

# 2. 예측 결과 (v4.0.0: pyarrow 메타데이터 에러 방지를 위해 fastparquet 명시)
if results.get('val_predictions') is not None:
    val_path = paths.get_seq_val_predictions()
    results['val_predictions'].to_parquet(val_path, engine='fastparquet', index=False)
    print(f"   ✅ val_predictions:  {val_path}")

if results.get('test_predictions') is not None:
    test_path = paths.get_seq_test_predictions()
    results['test_predictions'].to_parquet(test_path, engine='fastparquet', index=False)
    print(f"   ✅ test_predictions: {test_path}")

if should_save_csv(cfg, 3):
    csv_dir = paths.get_seq_csv_dir()
    ticker_name_map = load_ticker_name_map(paths)
    save_ticker_csv(results['test_predictions'], csv_dir, ticker_name_map,
                    desc="Saving seq test prediction CSVs")
    if results.get('val_predictions') is not None:
        save_ticker_csv(results['val_predictions'], csv_dir / "val", ticker_name_map,
                        desc="Saving seq val prediction CSVs")
else:
    print("⏭️  종목별 CSV 저장 비활성화 (output.save_csv.stage_03=false)")

# 3. 성능 요약
print(f"\n📊 성능 요약")
print(f"   검증  Avg RMSE: {results['valid_metrics']['avg_rmse']:.6f}")
print(f"   테스트 Avg RMSE: {results['test_metrics']['avg_rmse']:.6f}")
print(f"   검증  Avg IC:   {results['valid_metrics']['avg_ic']:.4f}")
print(f"   테스트 Avg IC:   {results['test_metrics']['avg_ic']:.4f}")

print(f"\n   Horizon별 테스트 RMSE (첫 5개):")
for h, metrics in list(results['test_metrics']['per_horizon'].items())[:5]:
    print(f"     h{h}: RMSE={metrics['rmse']:.6f}, IC={metrics['ic_mean']:.4f}")

print("\n✅ [Step 3c] 완료")
display(results['test_predictions'].head())

## 🏁 학습 완료

### ✅ 생성된 산출물
```
data/03_seq/{model_date}/gru/
├── weights.pt
├── config.json
├── val_predictions.parquet
└── test_predictions.parquet
```

### 🔁 다음 단계
- `config.yaml`의 `active_seq_model: "gru"`를 확인하고
- **`04_forecast_future.ipynb`**를 실행하면 GRU 전용 예측 경로가 자동 선택됩니다.
- 05단계는 Tabular / Seq 산출물을 동일한 인터페이스로 평가합니다.

### 📌 Tabular 트랙과 성능 비교
`03a_train_tabular.ipynb`의 test RMSE와 비교하여 두 트랙의 성능을 평가하십시오.  
예측 기간(forecast_horizon)이 다를 수 있으므로 같은 horizon 기준으로 비교합니다.